# 🛡️ Mangifera Shield — Cloud Training Notebook
**Train your Mango Disease Detection Model on free GPU!**

## Steps:
1.  **Runtime** → **Change runtime type** → select **T4 GPU**
2.  **Run All Cells** (Runtime → Run all)
3.  **Upload kaggle.json** when asked (for dataset download)
4.  **Download** the generated `mangifera_model_pack.zip` file
5.  **Extract** it into your project's `backend/` folder


In [ ]:
# @title 1. Setup Environment & Kaggle Auth
!pip install -q tensorflow==2.15.0 tensorflowjs==4.17.0 kaggle

import os
from google.colab import files

# Check for kaggle.json, if not present, ask to upload
if not os.path.exists('kaggle.json'):
    print("⚠️ Please upload your kaggle.json file (from Kaggle Account > Settings > API > Create New Token)")
    uploaded = files.upload()
    for fn in uploaded.keys():
        print(f'User uploaded file "{fn}" with length {len(uploaded[fn])} bytes')
    
    # Move to correct location
    !mkdir -p ~/.kaggle
    !cp kaggle.json ~/.kaggle/
    !chmod 600 ~/.kaggle/kaggle.json
    print("✅ Kaggle Auth configured!")
else:
    !mkdir -p ~/.kaggle
    !cp kaggle.json ~/.kaggle/
    !chmod 600 ~/.kaggle/kaggle.json
    print("✅ Kaggle Auth already present!")

In [ ]:
# @title 2. Download Dataset
import os

# Create directories
os.makedirs('dataset', exist_ok=True)
os.makedirs('model', exist_ok=True)
os.makedirs('tfjs_model', exist_ok=True)

print("⬇️ Downloading Mango Leaf Disease Dataset...")
!kaggle datasets download -d aman2000jaiswal/mango-leaf-disease-dataset -p dataset
!unzip -q dataset/mango-leaf-disease-dataset.zip -d dataset/
print("✅ Dataset downloaded and extracted!")

In [ ]:
# @title 3. Data Preprocessing & Augmentation
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
DATA_DIR = 'dataset' # extracted folder might be direct or nested, adjust if needed

# Check structure
if not os.path.exists(os.path.join(DATA_DIR, 'Anthracnose')):
    # Sometimes unzip puts it in a subfolder
    sub = [d for d in os.listdir(DATA_DIR) if os.path.isdir(os.path.join(DATA_DIR, d))][0]
    DATA_DIR = os.path.join(DATA_DIR, sub)
    print(f"Adjusted data dir to: {DATA_DIR}")

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest',
    validation_split=0.2
)

print("📦 Loading training data...")
train_ds = train_datagen.flow_from_directory(
    DATA_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training',
    shuffle=True
)

print("📦 Loading validation data...")
val_ds = train_datagen.flow_from_directory(
    DATA_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation',
    shuffle=False
)

class_indices = train_ds.class_indices
print(f"🏷️ Classes: {class_indices}")

import json
with open('model/class_labels.json', 'w') as f:
    json.dump({str(v): k for k, v in class_indices.items()}, f)


In [ ]:
# @title 4. Build & Train MobileNetV2 Model
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, Model, optimizers

# Base Model
base_model = MobileNetV2(input_shape=(224, 224, 3), include_top=False, weights='imagenet')
base_model.trainable = False

# Head
x = base_model.output
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(128, activation='relu')(x)
x = layers.Dropout(0.3)(x)
predictions = layers.Dense(len(class_indices), activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=predictions)

model.compile(optimizer=optimizers.Adam(learning_rate=0.001),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

print("🚀 Starting Training (Phase 1)... ")
history = model.fit(train_ds, validation_data=val_ds, epochs=15)

# Fine-tuning
print("🔧 Fine-tuning (Phase 2)...")
base_model.trainable = True
model.compile(optimizer=optimizers.Adam(1e-5), loss='categorical_crossentropy', metrics=['accuracy'])
history_fine = model.fit(train_ds, validation_data=val_ds, epochs=10)

# Save H5
model.save('model/disease_detector.h5')
print("✅ Model saved to model/disease_detector.h5")

In [ ]:
# @title 5. Convert to TF.js
import subprocess

print("🔄 Converting to TF.js format...")
cmd = "tensorflowjs_converter --input_format=keras --quantize_uint8 * model/disease_detector.h5 tfjs_model"
subprocess.run(cmd, shell=True)

print("✅ Converted! Files in tfjs_model:")
!ls tfjs_model

In [ ]:
# @title 6. Zip & Download Everything
import shutil
shutil.make_archive('mangifera_model_pack', 'zip', 'model')
# Also add tfjs
!zip -r mangifera_model_pack.zip tfjs_model/

from google.colab import files
files.download('mangifera_model_pack.zip')
print("🎉 Download started! Unzip this into your backend folder.")